# 4. Modelagem e Camada Gold

## 4.1 Definição da Modelagem

Nesta etapa será definida a estrutura dos dados que serão utilizados nas análises. A modelagem parte da tabela tratada na camada Silver e considera as perguntas de negócio definidas no início do projeto (item 1.3).

Como o conjunto de dados está concentrado em uma única tabela, em que cada registro representa uma solicitação de empréstimo, optou-se por manter essa granularidade na camada Gold, sem a criação de tabelas fato e dimensão. Para o escopo deste trabalho, essa separação aumentaria a quantidade de tabelas sem acrescentar informações necessárias às análises propostas.

A camada Gold será criada a partir dessa estrutura, mantendo os atributos necessários para as análises e acrescentando os campos derivados que forem necessários para responder às perguntas propostas.

## 4.2 Carregamento dos Dados da Camada Silver

Para iniciar a modelagem da camada Gold, será utilizada a tabela `loan_approval_silver`, criada e persistida na etapa anterior. Essa tabela contém os dados já tratados e será utilizada como ponto de partida para a preparação dos dados analíticos.

In [0]:
# --- Carregamento da tabela persistida na camada Silver ---

df_silver = spark.table("workspace.default.loan_approval_silver")

# --- Verificação do carregamento ---

print(f"Quantidade de registros: {df_silver.count()}")
print(f"Quantidade de atributos: {len(df_silver.columns)}")

display(df_silver.limit(10))

Quantidade de registros: 614
Quantidade de atributos: 13


Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
LP001002,Male,No,0,Graduate,No,5849,0.0,126.0,360,1,Urban,Y
LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360,1,Rural,N
LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360,1,Urban,Y
LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360,1,Urban,Y
LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360,1,Urban,Y
LP001011,Male,Yes,2,Graduate,Yes,5417,4196.0,267.0,360,1,Urban,Y
LP001013,Male,Yes,0,Not Graduate,No,2333,1516.0,95.0,360,1,Urban,Y
LP001014,Male,Yes,3+,Graduate,No,3036,2504.0,158.0,360,0,Semiurban,N
LP001018,Male,Yes,2,Graduate,No,4006,1526.0,168.0,360,1,Urban,Y
LP001020,Male,Yes,1,Graduate,No,12841,10968.0,349.0,360,1,Semiurban,N


### 4.2.1 Resultado do Carregamento dos Dados

A tabela `loan_approval_silver` foi carregada corretamente, mantendo os 614 registros e os 13 atributos definidos na etapa anterior.

A partir desses dados será feita a preparação da camada Gold, mantendo somente as transformações necessárias para responder às perguntas de negócio do projeto.

## 4.3 Preparação dos Dados para a Camada Gold

Nesta etapa serão criados os atributos necessários para analisar o comprometimento estimado da renda familiar.

A renda familiar será obtida pela soma da renda do solicitante com a renda do cônjuge. Como `LoanAmount` está expresso em milhares de unidades monetárias, seu valor será multiplicado por 1.000 antes do cálculo mensal, de forma a utilizar a mesma unidade monetária das variáveis de renda.

Também será calculado um valor mensal estimado do empréstimo, dividindo o valor solicitado pelo prazo da operação. Como a base não informa a taxa de juros, esse valor não representa a parcela real do empréstimo. O cálculo será utilizado apenas como uma estimativa para comparar o comprometimento da renda entre as solicitações aprovadas e reprovadas.




### 4.3.1 Criação dos Atributos para Análise

Serão criados os atributos `Renda_Familiar`, `Parcela_Mensal_Estimada` e `Percentual_Comprometimento_Renda` para complementar as informações disponíveis e permitir a análise proposta.

In [0]:
# --- Importação das funções utilizadas ---
from pyspark.sql import functions as F

# --- Criação dos atributos para análise ---

df_gold = (
    df_silver
    .withColumn(
        "Renda_Familiar",
        F.col("ApplicantIncome") + F.col("CoapplicantIncome")
    )
    .withColumn(
        "Parcela_Mensal_Estimada",
        F.round(
            (F.col("LoanAmount") * 1000) / F.col("Loan_Amount_Term"), 2
        )
    )
    .withColumn(
        "Percentual_Comprometimento_Renda",
        F.round(
            (F.col("Parcela_Mensal_Estimada") / F.col("Renda_Familiar")) * 100, 2
        )
    )
)

# --- Visualização dos atributos criados ---

display(
    df_gold.select(
        "Loan_ID",
        "ApplicantIncome",
        "CoapplicantIncome",
        "Renda_Familiar",
        "LoanAmount",
        "Loan_Amount_Term",
        "Parcela_Mensal_Estimada",
        "Percentual_Comprometimento_Renda",
        "Loan_Status"
    ).limit(10)
)

Loan_ID,ApplicantIncome,CoapplicantIncome,Renda_Familiar,LoanAmount,Loan_Amount_Term,Parcela_Mensal_Estimada,Percentual_Comprometimento_Renda,Loan_Status
LP001002,5849,0.0,5849.0,126.0,360,350.0,5.98,Y
LP001003,4583,1508.0,6091.0,128.0,360,355.56,5.84,N
LP001005,3000,0.0,3000.0,66.0,360,183.33,6.11,Y
LP001006,2583,2358.0,4941.0,120.0,360,333.33,6.75,Y
LP001008,6000,0.0,6000.0,141.0,360,391.67,6.53,Y
LP001011,5417,4196.0,9613.0,267.0,360,741.67,7.72,Y
LP001013,2333,1516.0,3849.0,95.0,360,263.89,6.86,Y
LP001014,3036,2504.0,5540.0,158.0,360,438.89,7.92,N
LP001018,4006,1526.0,5532.0,168.0,360,466.67,8.44,Y
LP001020,12841,10968.0,23809.0,349.0,360,969.44,4.07,N


In [0]:
# --- Verificação de renda familiar igual a zero ---

renda_zero = df_gold.filter(F.col("Renda_Familiar") == 0).count()

print(f"Registros com renda familiar igual a zero: {renda_zero}")

Registros com renda familiar igual a zero: 0


#### 4.3.1.1 Resultado da Criação dos Atributos

Os três novos atributos foram calculados para todos os registros. Também foi verificado se existiam casos com renda familiar igual a zero, o que poderia impedir o cálculo do percentual de comprometimento, mas nenhum registro apresentou essa condição.

Os valores de `Parcela_Mensal_Estimada` e `Percentual_Comprometimento_Renda` foram arredondados para duas casas decimais para facilitar sua leitura e interpretação nas análises.

### 4.3.2 Estrutura Final da Camada Gold

A estrutura final da camada Gold manterá os 13 atributos já disponíveis nos dados tratados e acrescentará os três atributos criados na etapa 4.3.1. Dessa forma, a tabela poderá ser utilizada para responder às perguntas de negócio sem eliminar informações que ainda possam ser úteis durante as análises.

Ao todo, a camada Gold será composta por 16 atributos, mantendo cada registro no nível de uma solicitação de empréstimo.

In [0]:
# --- Estrutura final da camada Gold ---

print(f"Quantidade de registros: {df_gold.count()}")
print(f"Quantidade de atributos: {len(df_gold.columns)}")

df_gold.printSchema()

Quantidade de registros: 614
Quantidade de atributos: 16
root
 |-- Loan_ID: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Married: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- Education: string (nullable = true)
 |-- Self_Employed: string (nullable = true)
 |-- ApplicantIncome: integer (nullable = true)
 |-- CoapplicantIncome: double (nullable = true)
 |-- LoanAmount: double (nullable = true)
 |-- Loan_Amount_Term: integer (nullable = true)
 |-- Credit_History: integer (nullable = true)
 |-- Property_Area: string (nullable = true)
 |-- Loan_Status: string (nullable = true)
 |-- Renda_Familiar: double (nullable = true)
 |-- Parcela_Mensal_Estimada: double (nullable = true)
 |-- Percentual_Comprometimento_Renda: double (nullable = true)



### 4.3.3 Validação dos Novos Atributos

Antes de persistir a camada Gold, será feita uma verificação dos atributos criados nesta etapa. O objetivo é conferir se os cálculos não geraram valores nulos e observar os valores mínimos e máximos encontrados.

In [0]:
# --- Validação dos atributos criados ---

colunas_gold = [
    "Renda_Familiar",
    "Parcela_Mensal_Estimada",
    "Percentual_Comprometimento_Renda"
]

resultado_validacao = []

for coluna in colunas_gold:
    resultado = df_gold.select(
        F.min(coluna).alias("Minimo"),
        F.max(coluna).alias("Maximo")
    ).first()

    nulos = df_gold.filter(F.col(coluna).isNull()).count()

    resultado_validacao.append(
        (coluna, float(resultado["Minimo"]), float(resultado["Maximo"]), nulos)
    )

# --- Criação da tabela para visualização ---

df_validacao_gold = spark.createDataFrame(
    resultado_validacao,
    ["Atributo", "Minimo", "Maximo", "Valores_Nulos"]
)

display(df_validacao_gold)

Atributo,Minimo,Maximo,Valores_Nulos
Renda_Familiar,1442.0,81000.0,0
Parcela_Mensal_Estimada,25.0,9250.0,0
Percentual_Comprometimento_Renda,0.7,123.69,0


#### 4.3.3.1 Resultado da Validação dos Novos Atributos

A validação não identificou valores nulos nos três atributos criados. A renda familiar variou entre 1.442 e 81.000, enquanto a parcela mensal estimada ficou entre 25 e 9.250.

O percentual de comprometimento da renda variou entre 0,70% e 123,69%. Valores acima de 100% foram mantidos, pois representam situações em que a parcela estimada supera a renda familiar informada. Esses casos serão considerados posteriormente na análise dos resultados.

## 4.4 Persistência da Camada Gold

Com a estrutura e os novos atributos validados, os dados serão persistidos em formato Delta para formar a camada Gold do pipeline.

In [0]:
# --- Persistência dos dados na camada Gold ---

(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.loan_approval_gold")
)

# --- Validação da tabela Gold persistida ---

df_gold_validacao = spark.table("workspace.default.loan_approval_gold")

print(f"Registros persistidos: {df_gold_validacao.count()}")
print(f"Atributos persistidos: {len(df_gold_validacao.columns)}")

display(df_gold_validacao.limit(10))

Registros persistidos: 614
Atributos persistidos: 16


Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,Renda_Familiar,Parcela_Mensal_Estimada,Percentual_Comprometimento_Renda
LP001002,Male,No,0,Graduate,No,5849,0.0,126.0,360,1,Urban,Y,5849.0,350.0,5.98
LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360,1,Rural,N,6091.0,355.56,5.84
LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360,1,Urban,Y,3000.0,183.33,6.11
LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360,1,Urban,Y,4941.0,333.33,6.75
LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360,1,Urban,Y,6000.0,391.67,6.53
LP001011,Male,Yes,2,Graduate,Yes,5417,4196.0,267.0,360,1,Urban,Y,9613.0,741.67,7.72
LP001013,Male,Yes,0,Not Graduate,No,2333,1516.0,95.0,360,1,Urban,Y,3849.0,263.89,6.86
LP001014,Male,Yes,3+,Graduate,No,3036,2504.0,158.0,360,0,Semiurban,N,5540.0,438.89,7.92
LP001018,Male,Yes,2,Graduate,No,4006,1526.0,168.0,360,1,Urban,Y,5532.0,466.67,8.44
LP001020,Male,Yes,1,Graduate,No,12841,10968.0,349.0,360,1,Semiurban,N,23809.0,969.44,4.07


### 4.4.1 Resultado da Persistência

A camada Gold foi persistida com sucesso em formato Delta, mantendo os 614 registros e os 16 atributos definidos durante a modelagem.

Após a gravação, a tabela foi carregada novamente para conferir o resultado da persistência e a disponibilidade dos dados para as próximas análises.

## 4.5 Catálogo de Dados

O catálogo de dados será utilizado para documentar e facilitar a consulta à estrutura construída ao longo do pipeline. A documentação será apresentada tanto pela estrutura técnica das tabelas registradas no Databricks quanto pelo dicionário dos atributos utilizados na camada Gold.

### 4.5.1 Catálogo Técnico no Databricks

As tabelas criadas nas camadas Bronze, Silver e Gold foram registradas no catálogo do Databricks dentro do schema `default`. Por meio do Catalog Explorer foi possível consultar as tabelas, seus atributos, tipos de dados e demais informações relacionadas à estrutura e ao armazenamento.

A verificação também permitiu confirmar que a tabela Gold está armazenada em formato Delta. As evidências dessa estrutura foram registradas por meio de capturas de tela da plataforma.

### 4.5.2 Dicionário de Dados

O dicionário de dados apresenta a descrição dos 16 atributos disponíveis na camada Gold, incluindo os atributos originais mantidos durante o tratamento e os três atributos criados durante a modelagem.

Além da descrição, serão apresentados o tipo de dado, o domínio ou unidade de cada atributo e sua origem, facilitando a compreensão da estrutura utilizada nas análises.

In [0]:
# --- Criação do dicionário de dados ---

dados_catalogo = [
    ("Loan_ID", "string", "Identificador único da solicitação", "Texto", "Base original"),
    ("Gender", "string", "Gênero do solicitante", "Male / Female", "Base original"),
    ("Married", "string", "Estado civil do solicitante", "Yes / No", "Base original"),
    ("Dependents", "string", "Quantidade de dependentes", "0 / 1 / 2 / 3+", "Base original"),
    ("Education", "string", "Nível de escolaridade", "Graduate / Not Graduate", "Base original"),
    ("Self_Employed", "string", "Indica se trabalha por conta própria", "Yes / No", "Base original"),
    ("ApplicantIncome", "integer", "Renda mensal do solicitante", "Unidades monetárias /mês", "Base original"),
    ("CoapplicantIncome", "double", "Renda mensal do cônjuge ou co-solicitante", "Unidades monetárias /mês", "Base original"),
    ("LoanAmount", "double", "Valor solicitado do empréstimo", "Milhares de unidades monetárias", "Base original"),
    ("Loan_Amount_Term", "integer", "Prazo do empréstimo", "Meses", "Base original"),
    ("Credit_History", "integer", "Indicador de histórico de crédito", "0 / 1", "Base original"),
    ("Property_Area", "string", "Área de localização do imóvel", "Rural / Semiurban / Urban", "Base original"),
    ("Loan_Status", "string", "Situação da solicitação", "Y / N", "Base original"),
    ("Renda_Familiar", "double", "Soma das rendas do solicitante e do cônjuge", "Unidades monetárias / mês", "Derivado"),
    ("Parcela_Mensal_Estimada", "double", "Valor mensal estimado do empréstimo sem juros", "Unidades monetárias / mês", "Derivado"),
    ("Percentual_Comprometimento_Renda", "double", "Percentual estimado da renda familiar comprometida", "Percentual (%)", "Derivado")
]

df_catalogo = spark.createDataFrame(
    dados_catalogo,
    ["Atributo", "Tipo", "Descricao", "Dominio_Unidade", "Origem"]
)

display(df_catalogo)

Atributo,Tipo,Descricao,Dominio_Unidade,Origem
Loan_ID,string,Identificador único da solicitação,Texto,Base original
Gender,string,Gênero do solicitante,Male / Female,Base original
Married,string,Estado civil do solicitante,Yes / No,Base original
Dependents,string,Quantidade de dependentes,0 / 1 / 2 / 3+,Base original
Education,string,Nível de escolaridade,Graduate / Not Graduate,Base original
Self_Employed,string,Indica se trabalha por conta própria,Yes / No,Base original
ApplicantIncome,integer,Renda mensal do solicitante,Valor monetário,Base original
CoapplicantIncome,double,Renda mensal do cônjuge ou co-solicitante,Valor monetário,Base original
LoanAmount,double,Valor solicitado do empréstimo,Milhares,Base original
Loan_Amount_Term,integer,Prazo do empréstimo,Meses,Base original


#### 4.5.2.1 Resultado do Dicionário de Dados

O dicionário reúne os 16 atributos da camada Gold, apresentando seus tipos, descrições, domínios ou unidades e respectivas origens.

Dos 16 atributos documentados, 13 são provenientes da base original e três foram derivados durante a preparação da camada Gold: `Renda_Familiar`, `Parcela_Mensal_Estimada` e `Percentual_Comprometimento_Renda`.

### 4.5.3 Linhagem das Tabelas

Para complementar o catálogo de dados, será documentada a linhagem das tabelas utilizadas no pipeline, identificando a origem de cada camada e as principais transformações realizadas entre Bronze, Silver e Gold.

In [0]:
# --- Criação da documentação de linhagem do pipeline ---

dados_linhagem = [
    (
        "loan_approval_bronze",
        "Bronze",
        "Dataset original",
        "Ingestão e persistência dos dados brutos, preservando a estrutura original da fonte"
    ),
    (
        "loan_approval_silver",
        "Silver",
        "loan_approval_bronze",
        "Remoção de coluna auxiliar, tratamento de valores nulos, padronização e adequação dos tipos de dados"
    ),
    (
        "loan_approval_gold",
        "Gold",
        "loan_approval_silver",
        "Criação de Renda_Familiar, Parcela_Mensal_Estimada e Percentual_Comprometimento_Renda"
    )
]

df_linhagem = spark.createDataFrame(
    dados_linhagem,
    ["Tabela", "Camada", "Origem", "Transformacoes_Principais"]
)

display(df_linhagem)

Tabela,Camada,Origem,Transformacoes_Principais
loan_approval_bronze,Bronze,Dataset original,"Ingestão e persistência dos dados brutos, preservando a estrutura original da fonte"
loan_approval_silver,Silver,loan_approval_bronze,"Remoção de coluna auxiliar, tratamento de valores nulos, padronização e adequação dos tipos de dados"
loan_approval_gold,Gold,loan_approval_silver,"Criação de Renda_Familiar, Parcela_Mensal_Estimada e Percentual_Comprometimento_Renda"


#### 4.5.3.1 Resultado da Linhagem das Tabelas

A linhagem evidencia o fluxo dos dados desde o dataset original até as camadas Bronze, Silver e Gold. A camada Bronze preserva os dados ingeridos em sua estrutura original, a Silver concentra os tratamentos e as padronizações realizadas, enquanto a Gold disponibiliza os atributos derivados utilizados nas análises.